In [ ]:
import ROOT
import os
import numpy as np
import pandas as pd

root_files = "/Titan0/aegis/pythia2/pythia8245/examples/Research/root_files_edit"


In [ ]:
# Viewer
signal_mode = "1000_2"
# print(os.path.exists("/Titan0/aegis/pythia2/pythia8245/examples/Research/root_files_edit/1000_2_events.root"))
chain = ROOT.TChain("Delphes")
chain.Add(f"{root_files}/{signal_mode}_events.root")

df = ROOT.RDataFrame(chain)
branches = [
        "SmallJet.Eta", "SmallJet.PT", "SmallJet.Phi", "SmallJet.TauTag", "Electron_size", "Muon_size", "MissingET.MET", "MissingET.Phi", "SmallJet.BTag",
        "FatJet.PT", "FatJet.Eta", "FatJet.Phi", "FatJet.Mass", "FatJet.Tau[5]", "FatJet.Mass" 
    ]

runtime_error: string ROOT::RDF::RInterfaceBase::GetColumnType(string_view column) =>
    runtime_error: TTree leaf FatJet.Tau[5] has both a leaf count and a static length. This is not supported.

Warning in <TClass::Init>: no dictionary for class HepMCEvent is available
Warning in <TClass::Init>: no dictionary for class Event is available
Warning in <TClass::Init>: no dictionary for class Weight is available
Warning in <TClass::Init>: no dictionary for class GenParticle is available
Warning in <TClass::Init>: no dictionary for class SortableObject is available
Warning in <TClass::Init>: no dictionary for class Track is available
Warning in <TClass::Init>: no dictionary for class Tower is available
Warning in <TClass::Init>: no dictionary for class Jet is available
Warning in <TClass::Init>: no dictionary for class MissingET is available
Warning in <TClass::Init>: no dictionary for class Electron is available
Warning in <TClass::Init>: no dictionary for class Photon is available
Warning in <TClass::Init>: no dictionary for class Muon is available
Warning in <TClass::Init>: no dictionary for class ScalarHT is available


In [ ]:
def make_dataset(period, run, c, reduce_root_file, outdir):
    os.makedirs(outdir, exist_ok=True)
    chain = ROOT.TChain("Delphes")
    chain.Add(reduce_root_file)
    df = ROOT.RDataFrame(chain)

    # branches = [
    #     "AnalysisJetsAuxDyn_eta", "AnalysisJetsAuxDyn_pt", "AnalysisJetsAuxDyn_NNJvtPass", "AnalysisJetsAuxDyn_phi",
    #     "AnalysisTauJetsAuxDyn_JetDeepSetTight", "AnalysisElectronsAuxDyn_DFCommonElectronsLHTight", "AnalysisMuonsAuxDyn_muonType", "AnalysisMuonsAuxDyn_quality",
    #     "MET_Core_AnalysisMETAuxDyn_mpx", "MET_Core_AnalysisMETAuxDyn_mpy", "MET_Core_AnalysisMETAuxDyn_sumet",
    #     "BTagging_AntiKt4EMPFlowAuxDyn_DL1dv01_pu", "BTagging_AntiKt4EMPFlowAuxDyn_DL1dv01_pc", "BTagging_AntiKt4EMPFlowAuxDyn_DL1dv01_pb",
    #     "AnalysisLargeRJetsAuxDyn_pt", "AnalysisLargeRJetsAuxDyn_eta", "AnalysisLargeRJetsAuxDyn_phi",
    #     "AnalysisLargeRJetsAuxDyn_m", "AnalysisLargeRJetsAuxDyn_Tau1_wta", "AnalysisLargeRJetsAuxDyn_Tau2_wta", "AnalysisLargeRJetsAuxDyn_Tau3_wta"
    # ]
    branches = [
        "SmallJet.Eta", "SmallJet.PT", "SmallJet.Phi", "SmallJet.TauTag", "Electron_size", "Muon_size", "MissingET.MET", "MissingET.Phi", "SmallJet.BTag",
        "FatJet.PT", "FatJet.Eta", "FatJet.Phi", "FatJet.Mass", "FatJet.Tau[5]", "FatJet.Mass" 
    ]

    data = df.AsNumpy(branches)

    outfile = os.path.join(outdir, f"dataset_{run}_{c}.txt")
    with open(outfile, "w") as fout:
        fout.write("pT_j1 eta_j1 phi_j1 pT_j2 eta_j2 phi_j2 m_jj "
                   "tau21_j1 tau21_j2 tau32_j1 tau32_j2 "
                   "met phi_met min_dPhi ht\n")

        n_events = len(data["AnalysisJetsAuxDyn_pt"])
        for i in range(n_events):
            # --- small-R jets (for cuts) ---

            jet_pt = np.array(data["AnalysisJetsAuxDyn_pt"][i]) / 1000.0
            jet_eta = np.array(data["AnalysisJetsAuxDyn_eta"][i])
            jet_phi = np.array(data["AnalysisJetsAuxDyn_phi"][i])
            rvec_jvt = data["AnalysisJetsAuxDyn_NNJvtPass"][i]
            jvt_pass = np.array([ord(rvec_jvt[j]) for j in range(len(rvec_jvt))], dtype=int)

            pu = np.array(data["BTagging_AntiKt4EMPFlowAuxDyn_DL1dv01_pu"][i])
            pc = np.array(data["BTagging_AntiKt4EMPFlowAuxDyn_DL1dv01_pc"][i])
            pb = np.array(data["BTagging_AntiKt4EMPFlowAuxDyn_DL1dv01_pb"][i])

            rvec_tau = data["AnalysisTauJetsAuxDyn_JetDeepSetTight"][i]
            tau_tight = np.array([ord(rvec_tau[j]) for j in range(len(rvec_tau))], dtype=int)

            rvec_elec = data["AnalysisElectronsAuxDyn_DFCommonElectronsLHTight"][i]
            electron_tight = np.array([ord(rvec_elec[j]) for j in range(len(rvec_elec))], dtype=int)

            rvec_muon = data["AnalysisMuonsAuxDyn_quality"][i]
            muon_Qual = np.array([ord(rvec_muon[j]) for j in range(len(rvec_muon))], dtype=int)
            muon_Type = np.array(data["AnalysisMuonsAuxDyn_muonType"][i])

            mpx = data["MET_Core_AnalysisMETAuxDyn_mpx"][i][0]
            mpy = data["MET_Core_AnalysisMETAuxDyn_mpy"][i][0]
            met = data["MET_Core_AnalysisMETAuxDyn_sumet"][i][0]/1000.0
            phi_met = np.arctan2(mpy, mpx)

            fat_pt = np.array(data["AnalysisLargeRJetsAuxDyn_pt"][i]) / 1000.0
            fat_eta = np.array(data["AnalysisLargeRJetsAuxDyn_eta"][i])
            fat_phi = np.array(data["AnalysisLargeRJetsAuxDyn_phi"][i])
            fat_m   = np.array(data["AnalysisLargeRJetsAuxDyn_m"][i]) / 1000.0
            tau1    = np.array(data["AnalysisLargeRJetsAuxDyn_Tau1_wta"][i])
            tau2    = np.array(data["AnalysisLargeRJetsAuxDyn_Tau2_wta"][i])
            tau3    = np.array(data["AnalysisLargeRJetsAuxDyn_Tau3_wta"][i])

            # --- Cuts ---
            if len(jet_pt) < 2: continue
            if np.sum(np.abs(jet_eta) < 2.8) < 2: continue
            if jet_pt[0] < 250 or jvt_pass[0] != 1: continue
            if jet_pt[1] < 30 or jvt_pass[1] != 1: continue
            dphis = [abs(ROOT.TVector2.Phi_mpi_pi(jphi - phi_met)) for jphi in jet_phi]
            if np.sum(np.array(dphis) < 2.0) <= 1: continue
            if np.sum(np.log(pb/(fc*pc + (1-fc)*pu)) > cut_77) >= 2: continue
            if np.sum(tau_tight == 1) > 0: continue
            if np.sum(electron_tight == 1) > 0: continue
            if np.sum(muon_Type == 0) > 0 and (np.sum((muon_Qual == 8) | (muon_Qual == 9) > 0)): continue
            if len(fat_pt) < 2: continue

            j1 = ROOT.TLorentzVector(); j1.SetPtEtaPhiM(fat_pt[0], fat_eta[0], fat_phi[0], fat_m[0])
            j2 = ROOT.TLorentzVector(); j2.SetPtEtaPhiM(fat_pt[1], fat_eta[1], fat_phi[1], fat_m[1])
            m_jj = (j1+j2).M()

            tau21_j1 = tau2[0]/tau1[0] if tau1[0] > 0 else -1
            tau21_j2 = tau2[1]/tau1[1] if tau1[1] > 0 else -1
            tau32_j1 = tau3[0]/tau2[0] if tau2[0] > 0 else -1
            tau32_j2 = tau3[1]/tau2[1] if tau2[1] > 0 else -1
            min_dPhi = np.min(dphis)
            ht = np.sum(jet_pt)

            row = [
                fat_pt[0], fat_eta[0], fat_phi[0],
                fat_pt[1], fat_eta[1], fat_phi[1],
                m_jj,
                tau21_j1, tau21_j2,
                tau32_j1, tau32_j2,
                met, phi_met, min_dPhi, ht
            ]
            fout.write(" ".join(map(str, row)) + "\n")
